# 01/ Exploratory Data Analysis (EDA) - CMAPSS Dataset

This notebook performs basics Exploratory Data Analysis (EDA) on the CMAPSS dataset to understand sensor behaviors, remove uninformative features, and detect multicollinearity.

### 1 - Setup

In this section, we manage system paths to ensure local source modules can be imported. We then load the core data manipulation, and visualization libraries required for the analysis.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_processing import load_train_data
import matplotlib.pyplot as plt
import seaborn as sns

### 2 - Dataset Loading

Here, we load the training subset FD001 using a custom function that compute the RUL variable and inspect the first few rows to verify the structural integrity of the dataframe.

In [ ]:
df = load_train_data("../CMAPSSData/train_FD001.txt")
df.head()

### 3 - Feature Selection via Variance Analysis

#### a) Variance Thresholding

Not all sensors provide useful predictive signals. Some may remain constant throughout the engine's cycles. We isolate the sensor columns, calculate their variance, and split them into two categories:
- High Variance Features.
- Low/Zero Variance Features: Constant signals that do not contribute predictive value.

In [ ]:
# Variance table for all sensors
variance_table = df[df.columns[5:26]]
variance_table = variance_table.var()
variance_table = variance_table.to_frame(name='Variance')
variance_table = variance_table.round(5)
print(variance_table)

# list of features with variance less than 0.01
low_variance_features = variance_table[variance_table['Variance'] == 0].index.tolist()
print(f"Features with low variance: {low_variance_features}")

# list of features with variance greater than 0.01
high_variance_features = variance_table[variance_table['Variance'] != 0].index.tolist()
print(f"Features with high variance: {high_variance_features}")

### 4 - Visualizing Sensor Behavior over Engine Cycles

To validate our variance-based filtering, we visualize how both feature subsets behave relative to the elapsed cycles of the engines.

#### a) High Variance Sensor Trends

Plotting a FacetGrid of scatter plots, we look for visible trajectories or trends. As observed below, these high-variance sensors change dynamically as cycles increase, making them highly informative for Remaining Useful Life (RUL) prediction.

In [ ]:
long_df = df.melt(
    id_vars=["cycle", "unit"],
    value_vars=high_variance_features,
    var_name="sensor",
    value_name="value"
)

g = sns.FacetGrid(
    long_df,
    col="sensor",
    col_wrap=2,
    height=3,
    aspect=1.2,
    sharey=False 
)

g.map_dataframe(
    sns.scatterplot,
    x="cycle",
    y="value",
    hue="unit",
    alpha=0.5,
    legend=False
)

g.set_axis_labels("Cycle", "Value")
g.set_titles("{col_name}")

plt.show()

#### b) Low Variance Sensor Trends (Candidate for Removal)

Conversely, plotting the low-variance features shows flat, static lines across all cycles. Because these sensors offer no patterns regarding engine degradation, they will be removed from analysis.

In [ ]:
long_df = df.melt(
    id_vars=["cycle", "unit"],
    value_vars=low_variance_features,
    var_name="sensor",
    value_name="value"
)

g = sns.FacetGrid(
    long_df,
    col="sensor",
    col_wrap=2,
    height=3,
    aspect=1.2,
    sharey=False 
)

g.map_dataframe(
    sns.scatterplot,
    x="cycle",
    y="value",
    hue="unit",
    alpha=0.5,
    legend=False
)

g.set_axis_labels("Cycle", "Value")
g.set_titles("{col_name}")

plt.show()

### 5 - Target Variable Analysis & Multicollinearity

#### a) Distribution of Maximum RUL per Engine

To understand the distribution of engine lifespans within the dataset, we calculate the maximum cycle reached by each engine unit. The resulting histogram displays a balanced, roughly Gaussian distribution.

In [ ]:
# Add the variable max_rul_per_engine to store the maximum RUL for each unit
max_rul = df.groupby("unit")["RUL"].max()

# Plot the distribution of max RUL per engine
plt.figure(figsize=(10, 6))
plt.hist(max_rul, bins=30, edgecolor='black', color='#301934')
plt.xlabel('Max RUL')
plt.ylabel('Nb. of Engines')
plt.title('Distribution of Max RUL per Engine')
plt.show()

#### b) Sensor Correlation Heatmap

Finally, we compute a Pearson correlation matrix for the high-variance sensors to identify redundant relationships. The heatmap reveals significant clusters of highly correlated variables.

In [ ]:
# Heatmap of sensor correlations to detect redundancy.
corr_matrix = df[high_variance_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0)
plt.title("Sensor Correlation Heatmap")
plt.show()